(course-core-the-form-agnostic-philosophy)=

# Module 1: The Form-Agnostic Philosophy

**Welcome, Apprentice Master.** Your journey into the heart of molecular matter starts here.

In this course, you are not just learning a software library; you are learning to **talk to atoms**. You are building your own virtual laboratory where the walls of file formats and software limitations simply do not exist.

The most important concept in **MolSysMT** is the **Form**. In traditional Computational Biology, you are often a prisoner of your file format: if you have a PDB file, you use one library; if you have an OpenMM object, you use another. 

MolSysMT breaks these chains. It treats every molecular data structure as a **Form**. Whether it is a file on disk, an object in memory, or a string of text, MolSysMT sees them all as "Molecular Systems".

```{admonition} Glossary: Form
:class: dropdown info
A **Form** is any specific way molecular data is stored or represented (e.g., a `.pdb` file, an `mdtraj.Trajectory` object, or a `molsysmt.MolSys` native system).
```

:::{info}
:class: dropdown
**Supported Forms Portal:** See the [full list of 60+ supported forms](https://www.uibcdf.org/molsysmt) in our User Guide Tools section.
:::

(course-core-the-form-agnostic-philosophy-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Define **Molecular Systems** and **Forms** in MolSysMT.
- Query data representations using `msm.get_form()`.
- Convert systems across forms using `msm.convert()`.
- Generate quick structural audits using `msm.info()`.
```

Before converting, you can request `msm.get_conversion_report()` to inspect detected information loss without creating the destination. The source can be any supported form. Check the report's `audited_scopes` and `is_exhaustive`: partial audit coverage does not guarantee preservation of every field or successful writing. See {ref}`Tutorial_Get_Conversion_Report` and {ref}`cookbook-native-sdf` for the workflow.

Form detection uses declared metadata, and conversion initializes the adapters
needed by its route. The first use of another form can still have an initialization
cost. Enumerating all callable capabilities may load more adapters; inspecting
an optional form does not enable execution without its required engine.


### 1. The Universal Sampler: `msm.get_form()`

Let's begin by importing MolSysMT and its built-in demonstration systems repository.

In [1]:
import molsysmt as msm
from molsysmt import systems

:::{tip}
:class: dropdown
**Import Convention:** The core developers of MolSysMT strongly recommend importing the package as `msm` (`import molsysmt as msm`).
:::

:::{note}
:class: dropdown
**Demo Systems Repository:** `molsysmt.systems` is a built-in catalog of curated demonstration and test molecular systems shipped directly with MolSysMT. See the {ref}`user-foundations-index` guide for more details.
:::

To demonstrate that MolSysMT doesn't care about the size or the source, let's look at four systems from our specialized paths. Notice how we use the same `msm.get_form()` function for all of them.

In [2]:
# 🔬 Path A: Alzheimer (Amyloid-Beta Fibril from PDB ID)
alzheimer_system = 'pdb_id:2BEG'

# ♻️ Path B: Enzyme Engineering (Triosephosphate Isomerase from internal database)
enzyme_system = systems['TcTIM']['1tcd.h5msm']

# 💊 Path C: Antiviral Hunter (Barnase-Barstar Complex)
antiviral_system = systems['Barnase-Barstar']['barnase_barstar.h5msm']

# ⚡ Path D: Nano-Mechanic (POPC Membrane)
membrane_system = systems['POPC membrane']['popc_membrane.dcd']

systems_list = [alzheimer_system, enzyme_system, antiviral_system, membrane_system]
for sys in systems_list:
    print(f"Form found: {msm.get_form(sys)}")

Form found: string:pdb_id
Form found: file:h5msm
Form found: file:h5msm
Form found: file:dcd


:::{hint}
:class: dropdown
**msm.get_form()**: Identifies and returns the structural or data form of a given molecular system item (e.g. `'string:pdb_id'`, `'file:h5msm'`, `'openmm.Topology'`). See API doc: {func}`molsysmt.basic.get_form`.
:::

A declared text prefix selects its own form detector before unrelated text
heuristics. For example, `pdbqt_text:` recognition does not load RDKit. This
identifies the representation; parsing and chemical validation happen later.
See {ref}`Tutorial_Get_form` for the distinction.


### 2. The Power of Conversion: `msm.convert()`

You are never stuck in a form. You can `convert()` any system into another. But here is the magic: while the **Form** changes, the **Content** remains invariant.

In [3]:
# Let's convert the Enzyme (H5MSM file) into an OpenMM Topology
openmm_topo = msm.convert(enzyme_system, to_form='openmm.Topology')

print(f"New form: {msm.get_form(openmm_topo)}")

New form: openmm.Topology


:::{hint}
:class: dropdown
**msm.convert()**: Converts a molecular system between any of the 89 supported forms in memory or on disk. See API doc: {func}`molsysmt.basic.convert`.
:::

:::{note}
:class: dropdown

A recognized form has a bounded conversion contract. For example, the native
SDF adapter reads one V2000/V3000 record without RDKit, retains source hydrogens
and explicit chemistry, including V3000 coordination roles. Legacy blank
version markers use V2000 syntax; declared integer-covalent valence retains
virtual H counts in ChemicalStates without inventing H atoms or coordinates.
Undeclared counts remain unknown. The adapter rejects
unsupported active stereo/query encodings by default. An explicit
`stereo_engine='rdkit'` opt-in interprets supported tetrahedral and double-bond
stereo with the optional provider; it does not remove source hydrogens. An SDF property block needs explicit
authorization to discard; recognition does
not promise complete information preservation. Read the
{ref}`native SDF recipe <cookbook-native-sdf>` before converting ligand files.

An unselected SDF identity copy checks the single-record envelope and retains
source bytes even outside the native chemical profile. An exact conversion
report then certifies preservation, not chemical validity or docking readiness.
Native conversion and selected copies still require supported interpretation.
:::

The experimental PDBQT file and explicit `pdbqt_text:` string adapters read
prepared AutoDock labels, charges and one structure without external toolkits.
The graph remains partial. A ligand's declared torsion tree stays separate;
full MolSys projection requires explicit `discard_torsion_tree=True`.
Writing requires supplied charges, types and `typing_scheme='autodock4'`;
no preparation or hydrogen merging is performed. See {ref}`cookbook-native-pdbqt`.

When you read a local `.cif.gz` file with `msm.convert()`, its source and
neighboring files remain unchanged. Each read has independent temporary storage,
so concurrent reads can share an input in a directory without write permission.
Temporary storage is cleaned after success or failure. This is eager parsing,
not structure streaming; see {ref}`the compressed CIF form <Tutorial_Form_file_cif_gz>`.

When converting a local PDB to MolSys or Topology, an explicit
`get_missing_bonds=False` keeps only file-declared connectivity and avoids
OpenMM inference. The same choice works through PDBFileHandler. The existing
file/handler default `True` attempts optional OpenMM inference and can leave
only declared edges when that legacy inference is unavailable or fails.
A successfully loaded structure is not a complete chemical assignment.
See {ref}`cookbook-pdb-connectivity-policy`.

A topology alone cannot supply PDB positions. For native topology export,
provide a length quantity on the full source atom axis; selection keeps the
atom names and coordinates aligned, and the native PDB writer retains multiple
structures. Text-to-MDTraj conversions manage their scratch files internally.
See {ref}`Tutorial_Convert` for input arguments and units.

The explicit RDKit SDF writer verifies accurate CIP labels and the isomeric
graph before opening its destination. Transient non-CIP marks introduced by
3D readback do not require a chemical repair; genuine/declared stereo, isotope
and hydrogen changes remain rejection conditions. Source coordinates and
chemical assignments are retained. See {ref}`cookbook-native-sdf` for the
bounded original-input controls and optional scientific attribution.


### 3. Quick Reports with `msm.info()`

How do we know the system is still the same after conversion? We use **`msm.info()`** to get a styled biological inventory.

In [4]:
# Report of the converted OpenMM object
msm.info(openmm_topo)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_proteins,n_structures
openmm.Topology,3983,662,167,4,167,3,165,2,None


Notice that the biological content is identical. `msm.info()` is your go-to tool for a quick human-readable audit.

:::{hint}
:class: dropdown
**msm.info()**: Displays a human-readable summary of a molecular system's topology, structures, and attributes. See API doc: {func}`molsysmt.basic.info`.
:::

## Inspecting capabilities as data

Use `msm.supported.forms(output_type='list')` to enumerate form names without
reading a rendering object's `.data`. The four `msm.supported` reports offer
`dataframe` and `dictionary` outputs while keeping styled tables as their default.
Dictionaries declare both axes with `index`, `columns` and row-major `data`.
Form/type/viewer filters and matrix orientation still apply. Reported conversion
routes are declarations; required attributes and optional providers must still
be available for execution. See {ref}`Tutorial_Capability_Reports`.


In [5]:
import pandas as pd
import molsysmt as msm
names = msm.supported.forms(output_type='list')
assert 'molsysmt.MolSys' in names
routes = msm.supported.conversions(
    from_form=['molsysmt.MolSys'], to_form=['molsysmt.Topology'],
    output_type='dataframe')
assert bool(routes.loc['molsysmt.MolSys', 'molsysmt.Topology'])
record = msm.supported.conversions(
    from_form=['molsysmt.MolSys'], to_form=['molsysmt.Topology'],
    output_type='dictionary')
pd.testing.assert_frame_equal(pd.DataFrame(**record), routes)
assert 'Syntax' in msm.supported.syntaxes(output_type='dataframe')
assert 'Library' in msm.supported.dependencies(output_type='dataframe')
print('Declared forms:', len(names))
print('MolSys to Topology:', bool(routes.iloc[0, 0]))


Declared forms: 96
MolSys to Topology: True


--- 

### 🏆 Challenge 1: The Form Hunter

1. Load the **T4 Lysozyme** training ground using strictly its PDB ID: `'pdb_id:181L'`.
2. Check its form using `msm.get_form()`.
3. Use `msm.info()` to see what's inside (How many protein molecules? How many waters?).
4. Convert it into a **Sequence string** (`to_form='string:amino_acids_3'`).

If you succeed, you are ready for **Module 2: Molecular Attributes**.

```{key-takeaway}
In MolSysMT, everything is a System. The Form is just the container; the information is the priority.
```

(course-core-the-form-agnostic-philosophy-see-also)=
Gzip-compressed PDB inputs are recognized as `file:pdb.gz`. You use the same
`msm.convert()` boundary and choose the native target and connectivity policy,
without creating a decompressed file. The adapter is currently an experimental
input route with eager parsing. See {ref}`Tutorial_Form_file_pdb_gz` for an
executed example preserving original archive bytes.

:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.basic.get_form` — Form identification engine.
- {func}`molsysmt.basic.convert` — Form conversion engine.
- {func}`molsysmt.basic.info` — Biological inventory reporter.

**Related Course Modules & Guides:**
- Next Module: [Module 2: Molecular Attributes](../Common_Core/02_Molecular_Attributes.ipynb)
- User Guide: {ref}`user-foundations-index`
:::